In [20]:
import sqlite3
import pandas as pd
import json, os

db_path = os.path.join("rdguru.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()


In [21]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS summary (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pmid TEXT,
    orpha TEXT,
    disease_name TEXT,
    file_name TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS case_text_record (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pmid TEXT,
    orpha TEXT,
    disease_name TEXT,
    text_report TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS phenotypes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    identifier TEXT,
    age TEXT,
    sex TEXT,
    hpo_terms TEXT
)
""")
conn.commit()


In [ ]:
#LOAD and insert summary data

summary_csv = os.path.join("summary.csv")
summary_df = pd.read_csv(summary_csv, sep=",")
summary_df = summary_df.rename(columns={
    "PMID": "pmid",
    "File Name": "file_name",
    "ORPHA": "orpha",
    "Disease Name": "disease_name"
})
summary_df = summary_df[["pmid", "orpha", "disease_name", "file_name"]]
summary_df.to_sql("summary", conn, if_exists="append", index=False)
print(f"✅ Inserted {len(summary_df)} summary rows")


✅ Inserted 238 summary rows


In [26]:
#load and insert text data

text_csv = os.path.join("text_data.csv")
text_df = pd.read_csv(text_csv , encoding='latin1')
text_df = text_df.rename(columns={
    "pmid": "pmid",
    "orpha": "orpha",
    "disease name": "disease_name",
    "text": "text_report"
})
text_df = text_df[["pmid", "orpha", "disease_name", "text_report"]]
text_df.to_sql("case_text_record", conn, if_exists="append", index=False)
print(f"✅ Inserted {len(text_df)} text-report rows")


✅ Inserted 102 text-report rows


In [28]:
#Load and insert case info

case_info_path = os.path.join("/content/drive/MyDrive/rdguru-test-main/case_info")

phenotype_rows = []
for file in os.listdir(case_info_path):
    if file.endswith(".json"):
        fpath = os.path.join(case_info_path, file)
        with open(fpath) as f:
            data = json.load(f)
        identifier = data.get("id")
        sex = data.get("subject", {}).get("sex", "")
        age = data.get("subject", {}).get("ageAtCollection", {}).get("age", "")
        hpo_terms = [feat["type"]["id"] for feat in data.get("phenotypicFeatures", [])]
        phenotype_rows.append({
            "identifier": identifier,
            "sex": sex,
            "age": age,
            "hpo_terms": json.dumps(hpo_terms)
        })

phenotypes_df = pd.DataFrame(phenotype_rows)
phenotypes_df.to_sql("phenotypes", conn, if_exists="append", index=False)
print(f"✅ Inserted {len(phenotypes_df)} phenotype rows")


✅ Inserted 238 phenotype rows


In [29]:
for table in ["summary", "case_text_record", "phenotypes"]:
    print(f"\n=== {table.upper()} ===")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 3", conn))



=== SUMMARY ===


,id,pmid,orpha,disease_name,file_name
0,1,28832562,77298,Anophthalmia/microphthalmia-esophageal atresia...,204301
1,2,28832562,1340,Cardiofaciocutaneous syndrome,204303
2,3,28832562,634,Netherton syndrome,204317



=== CASE_TEXT_RECORD ===


,id,pmid,orpha,disease_name,text_report
0,1,20398363,ORPHA:79320,ALG6-CDG,A 9 year old Saudi boy (Fig. 1) was referred f...
1,2,21129173,ORPHA:169186,Autosomal recessive centronuclear myopathy,Patient 1 is a 13 year old girl belonging to a...
2,3,21129173,ORPHA:169186,Autosomal recessive centronuclear myopathy,"Patient 2, a 14 year old boy, is the first-deg..."



=== PHENOTYPES ===


,id,identifier,age,sex,hpo_terms
0,1,20388228,P17Y,FEMALE,"[""HP:0010463"", ""HP:0000151"", ""HP:0010885"", ""HP..."
1,2,19232125,P36Y,MALE,"[""HP:0006323"", ""HP:0000164"", ""HP:0002653"", ""HP..."
2,3,20398363,P9Y,MALE,"[""HP:0001344"", ""HP:0008689"", ""HP:0000248"", ""HP..."


In [30]:
# Find all cases with 'Pompe' in disease name
query = """
SELECT s.pmid, s.disease_name, t.text_report
FROM summary s
JOIN case_text_record t ON s.orpha = t.orpha
WHERE s.disease_name LIKE '%Pompe%'
"""
pompe_cases = pd.read_sql_query(query, conn)
display(pompe_cases)

# Retrieve phenotypes for a given identifier
identifier = "0204301"
q2 = f"SELECT * FROM phenotypes WHERE identifier='{identifier}'"
display(pd.read_sql_query(q2, conn))


,pmid,disease_name,text_report


,id,identifier,age,sex,hpo_terms
0,76,0204301,P9M,MALE,"[""HP:0006989"", ""HP:0001263"", ""HP:0000568"", ""HP..."
